# Activity: Data wrangling with Python

**Initial Due Date: 2026-01-09 10:00AM**  
**Final Due Date: 2026-01-12 4:15PM**

## Learning Objectives

By the end of this notebook, you will be able to:

1.  Prepare analysis-ready data from real-world source using Pandas
2.  Perform exploratory data visualization and analysis using Matplotlib and Seaborn
3.  Annotate data from external sources to enrich visualizations and analyses

## Introduction

The U.S. Energy Information Administration (EIA) publishes data on [weekly retail gasoline and diesel prices across the United States](https://www.eia.gov/dnav/pet/pet_pri_gnd_dcus_nus_w.htm). The data is derived from surveys of fuel retailers and is available back to 1993! Check out the online documentation for more details on specific terms, i.e., conventional vs. reformulated gasoline, ultra low sulfur vs. low sulfur diesel, etc. In this activity, we will wrangle and visualize this data using Pandas, Matplotlib and Seaborn. <span class="column-margin margin-aside">Adapted from a [TidyTuesday](https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-07-01/readme.md) dataset.</span>

Get started with the necessary imports:

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

## Part A: Data loading and wrangling

We have frequently loaded data from CSV files (comma-separated values), but often data of interest is (only) available in Excel or other formats. Fortunately Pandas supports loading Excel files ([and many other formats](https://pandas.pydata.org/docs/reference/io.html)). Unlike CSV files, Excel files may contain multiple sheets so we need to specify which sheet to load. We first manually download the Excel file to identify the relevant sheet, i.e., “Data 1”, and how many, if any rows or columns, we want to skip to just load the data of interest (2, in this case).

In [ ]:
url = "https://www.eia.gov/dnav/pet/xls/PET_PRI_GND_DCUS_NUS_W.xls"
# Load sheet named "Data 1", skipping first 2 rows
fuel = pd.read_excel(url, sheet_name="Data 1", header=2)
fuel.head()

### Exercise A1

The data is not tidy. For example, the fuel type data is spread across multiple columns. Before writing any code, add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly sketch out a plan to tidy this data. Our goal is to analyze trends by type (“gasoline” vs. “diesel”), grade (“all” vs. “regular” vs. “midgrade” vs. “premium” for gasoline and “ultra_low_sulfur” vs. “low_sulfur” for diesel) and formulation (“all” vs. “conventional” vs. “reformulated” for gasoline, diesel doesn’t have a formulation). In your plan identify any type conversions needed, i.e., what columns should be dates, categorical, etc.

*[TODO: Your response here]*

### Exercise A2

Perform the data tidying steps you outlined above to produce a Pandas `DataFrame` named `fuel_tidy` with precisely typed “date”, “type”, “grade”, “formulation” and “price” columns. Drop any rows with missing price data as those are not useful for our analyses. *The automated tests will look for the specified variable name, column names, types and levels (e.g., “ultra_low_sulfur” for the diesel grade). Make sure your names and values match those listed below.*

| Column name | Levels (if applicable) |
|--------------------------|----------------------------------------------|
| date |  |
| type | “gasoline”, “diesel” |
| grade | “all”, “regular”, “midgrade”, “premium” for gasoline; “ultra_low_sulfur”, “low_sulfur” for diesel |
| formulation | “all”, “conventional”, “reformulated” for gasoline; NA (NaN) for diesel |
| price |  |

Some suggestions:

-   Think about how to rename your columns to facilitate splitting them later. Recall we can readily split a string column into multiple columns using a delimiter.
-   While your notebook should ultimately run end-to-end without user intervention, the development process may be more manual and iterative. For example, it is OK to create a hard-coded dictionary for renaming the fuel descriptors (we don’t expect the column names in the upstream data to change often). You might execute code during development, e.g., `fuel.columns`, to get the information you need to create that dictionary.
-   It is OK to copy and paste, but be careful where you copy from. The nicely formatted column names in the rendered table view may differ from the actual column names in the `DataFrame` (spoiler alert, there may be multiple spaces before the parentheses in the actual string). You can use `fuel.columns` to get a list of the actual column names as strings.

In [ ]:
# TODO: Your code here
fuel_tidy.head()

## Part B: Exploratory visualization and analysis

### Exercise B1

Generate a line plot showing weekly retail prices over time for gasoline and diesel (for the “all” grade and formulation). Ensure your plot is clearly labeled (i.e. has appropriate axis labels, title, and legend).

This plot only uses a subset of the data, so you will need to filter `fuel_tidy` accordingly. Recall that you can use a boolean vector with Pandas’ square brackets to filter rows. What boolean expression describes your desired data? The element-wise logical operators `&` (and), `|` (or) and `~` (not) can be used to construct multi-input conditions. To ensure the desired operator precedence, wrap each condition in parentheses, e.g., `(fuel_tidy["grade"] == "all") & (fuel_tidy["formulation"] == "all")`.

Seaborn will automatically aggregate multiple data points for the same x-value (date) when using line plots (displaying the mean with the default confidence interval). That is *not intended* here, so if you see those semi-transparent confidence intervals, you likely have not filtered the data sufficiently/correctly.

In [ ]:
# TODO: Your code here
plt.show()

### Exercise B2

You likely observe substantial variability linked to world events (we will return to that below). But we also expect some seasonal variability (since people drive more during the summer). We have too much data to readily see seasonal trends. Generate a new version of your plot, for just the years 2023-2024, to better observe seasonal trends. Much like we use `str` as the accessor for string methods on a Pandas `Series`, we use `dt` as the accessor for date/time methods. For example, to extract the year from a date column, we can use `fuel_tidy["date"].dt.year`.

In [ ]:
# TODO: Your code here
plt.show()

### Exercise B3

Perform exploratory analysis to identify the most expensive months. Define a new `DataFrame` named `monthly_max` with columns “year”, “type”, “grade”, “formulation”, and “month” that reports the month with the highest weekly retail price for each combination of year, fuel type, grade and formulation.

Some suggestions:

-   Recall that we can group by multiple columns and/or series using a list of column/series names. Each level of the group can be either a string column name or a `Series` (e.g., the year derived from the “date” column).
-   Think about your choices for the some of the optional arguments to `groupby`, e.g., `observed`. Do you want to include all possible combinations of the grouping columns? How do you want to handle missing values in the grouping variables?
-   Recall that `reset_index` converts a `Series` with a multi-index into a `DataFrame` with columns corresponding to the index levels.
-   There are multiple ways to approach this problem. Any reasonable approach will be accepted. One idea is to sort the data by price in descending order, then group by the relevant columns and take the first row of each group. Another approach is to use the `idxmax()` aggregation to identify the index of the row with the maximum price within each group, then use that index to look up the corresponding month. Keep in mind we don’t want the maximum price itself, i.e., the `max()` aggregation, but rather the month in which that maximum price occurred. The `idxmax()` aggregator will return a `Series` of row labels with a multi-index corresponding to the grouping columns. You can use the `apply` method on that `Series` to invoke a function on each value in the `Series`. How could you define a function that would transform a row label (index) to the corresponding month?

In [ ]:
# TODO: Your code here

Use the following code to make a quick histogram visualization of this data. Here we will use Seaborn’s figure-level `displot` to create a “facet-ed” figure with separate subplots for gasoline and diesel. We use the `col` parameter to specify sublots for different “type”, and the `hue` parameter to color the bars of each histogram by fuel grade.

In [ ]:
# Return FacetGrid object with histograms of most expensive months
g = sns.displot(
    data=monthly_max,
    col="type", x="month", hue="grade",
    discrete=True,  # x is discrete (months), sets binwidth=1 and centers bin on values
    element="step", fill=False,  # Layer histograms as outlines for visibility
)
g.set(xlabel="Month", ylabel="Count")
g.set_titles(col_template="Fuel Type: {col_name}") # Provide a 'template' since each column has a different value
g.legend.set_title("Fuel Grade")
plt.show()

## Exercise B4

Add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly summarize your observations about seasonal fuel prices in a few sentences. A satisfactory answer will note any trends you observe, e.g., which months or seasons tend to be most expensive, any differences between gasoline and diesel, and provide possible hypotheses that explain those trends.

*[TODO: Your response here]*

## Part C: Annotation

We know that oil prices, and thus retail fuel prices, are influenced by world events. For example, the Russian invasion of Ukraine in early 2022 led to substantial increases in oil prices due to supply concerns.

### Exercise C1

Identify at least three other world events that likely influenced oil prices during the time period covered by this dataset (1993-present). Annotate your line plot of weekly retail prices over time with vertical lines and text labels to indicate those events (you should have 4 total, including the Ukraine invasion). You can use Matplotlib’s `axvline` method to add vertical lines to an existing plot. You can use the `text` method to add text annotations. You may need to experiment with the x,y positions and alignment of the text to ensure they are clearly visible.

In [ ]:
# TODO: Your code here
plt.show()

### Exercise C2

Add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly summarize why you think your selected events influenced oil prices.

*[TODO: Your response here]*

## Collaboration statement

In a new text cell immediately below this paragraph (or by editing this text cell to add a paragraph), briefly list who or what you collaborated with and how. Cite any sources here or with relevant inline comments in your code. Acknowledge all contributors, both people and AI, and what portions of this notebook they contributed. You do not need to cite or acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant assignment on [Gradescope](https://gradescope.com) via the “Upload option” (guide [here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)). **Both files must be uploaded at the same time and the file names must match the specification exactly for the autotesting to run successfully.**

1.  `activity_python_wrangling.ipynb`: Your completed IPython notebook. You can obtain this via the “File→Download→Download .ipynb” menu option in Colab.
2.  `activity_python_wrangling.py`: Your completed IPython notebook as a Python file. You can obtain this via the “File→Download→Download .py” menu option in Colab. This file is used to provide line-level feedback on your submission.

You can submit multiple times, with only the most recent submission (before the final due date) assessed for credit. Gradescope will run a series of automated unit tests on your notebook (which may takes 10s of seconds depending on the complexity of the notebook). Note that the tests performed by Gradescope are limited. Passing all of the visible tests does not guarantee that your submission correctly satisfies all of the requirements of the assignment.